In [1]:
import os
import sys
import shutil
import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torch.nn.functional as F


from tqdm import tqdm, trange

import warnings


from ml_collections import ConfigDict

In [2]:
sys.path.append('..')

In [11]:
# from CRT_utils.CRT.core.config import create_config, save_config
from CRT_utils.CRT.littlehelper import *

# from CRT_utils.CRT.test import test
# from CRT_utils.CRT.test_zero_context_zero_target_with_bbox import test
from CRT_utils.CRT.test_normal_context_normal_target_with_bbox import test

# from CRT_utils.CRT.test_visual_search import test
# from utils.evaluate_uncertainty import evaluate_uncertainty
from CRT_utils.CRT.core.config import create_config, save_config
from CRT_utils.CRT.core.dataset import COCODataset, COCODatasetWithID, COCODatasetMixOR, COCODatasetFullMix
# from CRT_utils.CRT.core.metrics import AccuracyLogger 


from CRT_utils.CRT.core.model_bbox_PE import Model

In [19]:
config_dict = ConfigDict()


config_dict['config']                = None
config_dict['outdir']                = '../CRT_utils/CRT_weights_and_config/CRT_COCO_random/CRT bbox PE test/'
config_dict['checkpoint']            = '../CRT_utils/CRT_weights_and_config/CRT_COCO_random/CRT bbox PE/checkpoint_21.tar'

config_dict['annotations']           = '../datasets/COCO18_dset_for_CRT_training/train_metadata.json'
config_dict['imagedir']              = '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt'

config_dict['test_annotations']      = '../datasets/COCO18_dset_for_CRT_training/filtered_val_metadata.json'
# config_dict['test_annotations']      = '../datasets/COCO18_dset_for_CRT_training/val_metadata.json'
config_dict['test_imagedir']         = '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt_test'
config_dict['test_frequency']        = 1

config_dict['epochs']                = 50
config_dict['save_frequency']        = 1
config_dict['print_batch_metrics']   = None

config_dict['batch_size']            = 32 
config_dict['learning_rate']         = None
config_dict['imbalance_reweighting'] = None
config_dict['num_decoder_heads']     = None
config_dict['num_decoder_layers']    = 6
config_dict['uncertainty_gate_type'] = None
config_dict['uncertainty_threshold'] = 0
config_dict['weighted_prediction']   = None

In [20]:
cfg = create_config(config_dict)

In [21]:
NUM_CLASSES     = 18
cfg.num_classes = NUM_CLASSES

In [22]:
model = Model.from_config(cfg)

In [23]:
checkpoint = torch.load(cfg.checkpoint, map_location="cpu")

device = (
    "cuda" if torch.cuda.is_available()
    else "mps"  # macbook uses metal performance shaders to GPU accelearation
    if torch.backends.mps.is_available()
    else "cpu"
)

model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)

Model(
  (context_encoder): Encoder(
    (encoder): Sequential(
      (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu0): ReLU(inplace=True)
      (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (denseblock1): _DenseBlock(
        (denselayer1): _DenseLayer(
          (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu2): ReLU(inplace=True)
          (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (denselayer2): _DenseLayer(
          (norm1): BatchNorm2d(96, eps=1e-05, momentum

In [24]:
test_accuracy = test(
    model, 
    cfg.test_annotations, 
    cfg.test_imagedir, 
#             '../datasets/COCO18_dset_for_CRT_training/category_idx_dict.pkl',
    outdir = config_dict.outdir,
    epoch  = 0,
)

-------------------------------
Annotation Counts
-------------------------------
potted plant                151
tv                          195
bottle                      187
chair                       725
car                         362
clock                        74
cup                         224
fork                        100
knife                        82
bowl                        270
toilet                      156
laptop                      174
mouse                        20
keyboard                    100
microwave                    29
oven                        116
sink                        107
stop sign                    34
Total                      3106
-------------------------------



Test Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 3106/3106 [03:16<00:00, 15.78it/s]


Total Test Accuracy: 0.8785407543182373
Class                Accuracy  
car                      0.9392
stop sign                1.0000
bottle                   0.8342
cup                      0.8170
fork                     0.8600
knife                    0.8537
bowl                     0.8630
chair                    0.8772
potted plant             0.9603
toilet                   0.8590
tv                       0.8923
laptop                   0.8736
mouse                    0.8000
keyboard                 0.9500
microwave                0.8276
oven                     0.8103
sink                     0.8505
clock                    0.9459
